
# Model Explainability and Scenario Analysis

## 1. Objective
This notebook examines how the existing agricultural yield prediction model produces predictions and explores hypothetical input scenarios to understand model behavior.

## 2. Key Questions
- Which features contribute most to the existing model's predictions?
- How do predictions vary across different input values?
- Are model explanations consistent with the model's learned behavior?
- How sensitive are predictions to hypothetical changes in selected features?
- Which conclusions are supported by the available model artifacts?

## 3. Methodology
- Locate and inspect existing trained-model artifacts and their associated feature definitions.
- Identify the model type, prediction target, and preprocessing requirements.
- Apply suitable explainability techniques based on the actual model and available dependencies.
- Evaluate hypothetical scenarios using valid feature inputs.
- Record assumptions, limitations, and reproducibility information.

## 4. Important Distinctions
Feature importance describes model behavior; it does not establish causation.
Hypothetical scenarios are model-based experiments, not real-world forecasts.
A prediction is only as reliable as the model, its training data, and the validity of the input features.

## 5. Constraints
- Reuse existing project artifacts and datasets.
- Do not retrain models unless explicitly required and justified.
- Do not invent feature names, model performance, or explanation results.
- Avoid extrapolating beyond the model's supported input range.
- If the existing artifacts are insufficient for an explanation, document the limitation rather than fabricate results.

## 6. Expected Outputs
Explainability results, scenario-comparison tables where supported, validation checks, and metadata documenting the analysis and its limitations.


In [1]:

from pathlib import Path
import json
import pandas as pd
import numpy as np

PROJECT_ROOT = Path(
    r"C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence"
)

DATA_PATH = (
    PROJECT_ROOT
    / "data/processed/weather_regional/apy_weather_integrated_exploratory.csv"
)

OUTPUT_DIR = PROJECT_ROOT / "models/model_explainability_and_scenarios"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Inspect existing model-related artifacts without loading or retraining them.
search_roots = [
    PROJECT_ROOT / "models",
    PROJECT_ROOT / "artifacts",
]

model_extensions = {".pkl", ".joblib", ".sav", ".onnx", ".json"}
model_files = []
metadata_files = []

for root in search_roots:
    if root.exists():
        for path in root.rglob("*"):
            if path.is_file():
                if path.suffix.lower() in model_extensions:
                    model_files.append(path)
                if path.name.lower() in {
                    "metadata.json",
                    "model_metadata.json",
                    "feature_names.json",
                    "model_config.json",
                }:
                    metadata_files.append(path)

print("Project root exists:", PROJECT_ROOT.exists())
print("Source dataset exists:", DATA_PATH.exists())
print("Model artifact candidates:", len(model_files))
print("Metadata files found:", len(metadata_files))

print("\nModel artifact candidates:")
for path in sorted(model_files):
    print("-", path.relative_to(PROJECT_ROOT))

print("\nMetadata files:")
for path in sorted(set(metadata_files)):
    print("-", path.relative_to(PROJECT_ROOT))

print("\nOutput directory:", OUTPUT_DIR)


Project root exists: True
Source dataset exists: True
Model artifact candidates: 8
Metadata files found: 0

Model artifact candidates:
- models\agricultural_risk_intelligence\agricultural_risk_metadata.json
- models\crop_recommendation\crop_recommendation_evaluation.json
- models\crop_recommendation\crop_recommendation_metadata.json
- models\weather_intelligence\low_yield_model_evaluation.json
- models\weather_intelligence\weather_intelligence_metadata.json
- models\yield_prediction\individual_crop_hgb_location.joblib
- models\yield_prediction\individual_crop_hgb_location_metadata.json
- models\yield_prediction\yield_model_deployment_readiness_report.json

Metadata files:

Output directory: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\models\model_explainability_and_scenarios


In [2]:

MODEL_DIR = PROJECT_ROOT / "models/yield_prediction"

MODEL_PATH = MODEL_DIR / "individual_crop_hgb_location.joblib"
MODEL_METADATA_PATH = (
    MODEL_DIR / "individual_crop_hgb_location_metadata.json"
)
READINESS_PATH = (
    MODEL_DIR / "yield_model_deployment_readiness_report.json"
)

def inspect_json(path):
    print(f"\n{'=' * 70}")
    print(path.name)
    print("=" * 70)

    if not path.exists():
        print("FILE NOT FOUND")
        return None

    with open(path, "r", encoding="utf-8") as file:
        content = json.load(file)

    print(json.dumps(content, indent=2, default=str))
    return content

model_metadata = inspect_json(MODEL_METADATA_PATH)
readiness_report = inspect_json(READINESS_PATH)

print("\n" + "=" * 70)
print("MODEL FILE CHECK")
print("=" * 70)
print("Model exists:", MODEL_PATH.exists())

if MODEL_PATH.exists():
    print("Model size (MB):", round(MODEL_PATH.stat().st_size / (1024 ** 2), 2))



individual_crop_hgb_location_metadata.json
{
  "model_name": "HistGradientBoostingRegressor",
  "project": "BharatCrop Intelligence",
  "candidate_scope": "Individual Crop Candidate cohort",
  "features": [
    "Crop",
    "Season",
    "Crop_Category",
    "State",
    "District",
    "FE_YearStart"
  ],
  "categorical_features": [
    "Crop",
    "Season",
    "Crop_Category",
    "State",
    "District"
  ],
  "numeric_features": [
    "FE_YearStart"
  ],
  "target": "Yield",
  "training_rows": 35784,
  "training_year_start": 2013,
  "training_year_end": 2022,
  "holdout_rows": 7369,
  "holdout_evaluated_during_final_fit": false,
  "validation_windows": [
    {
      "train_through": 2018,
      "validation_years": [
        2019,
        2020
      ]
    },
    {
      "train_through": 2020,
      "validation_years": [
        2021,
        2022
      ]
    }
  ],
  "validation_metrics": {
    "2019-2020": {
      "MAE": 448.306,
      "RMSE": 672.961,
      "R2": 0.798
    },
   

In [3]:

import joblib

# Load the existing trained model; do not retrain.
model = joblib.load(MODEL_PATH)

FEATURES = model_metadata["features"]
TARGET = model_metadata["target"]

print("Model type:", type(model).__name__)
print("Expected features:", FEATURES)
print("Prediction target:", TARGET)

# Inspect the model's exposed feature information, if available.
print("\nModel feature count:", getattr(model, "n_features_in_", "Not exposed"))

if hasattr(model, "feature_names_in_"):
    print("Model feature names:", list(model.feature_names_in_))
else:
    print("Model feature names: Not exposed directly")

# Inspect pipeline steps when this artifact is a pipeline.
if hasattr(model, "named_steps"):
    print("\nPipeline steps:")
    for step_name, step in model.named_steps.items():
        print(f"- {step_name}: {type(step).__name__}")
elif hasattr(model, "steps"):
    print("\nPipeline steps:")
    for step_name, step in model.steps:
        print(f"- {step_name}: {type(step).__name__}")
else:
    print("\nNo pipeline steps exposed by the loaded object.")

assert len(FEATURES) == 6, "Unexpected feature contract."
assert TARGET == "Yield", "Unexpected prediction target."

print("\nFeature contract check passed.")


Model type: Pipeline
Expected features: ['Crop', 'Season', 'Crop_Category', 'State', 'District', 'FE_YearStart']
Prediction target: Yield

Model feature count: 6
Model feature names: ['Crop', 'Season', 'Crop_Category', 'State', 'District', 'FE_YearStart']

Pipeline steps:
- preprocessor: ColumnTransformer
- model: HistGradientBoostingRegressor

Feature contract check passed.


In [4]:

preprocessor = model.named_steps["preprocessor"]
regressor = model.named_steps["model"]

print("Preprocessor type:", type(preprocessor).__name__)
print("Regressor type:", type(regressor).__name__)

print("\nColumn transformer configuration:")
for transformer_name, transformer, columns in preprocessor.transformers_:
    print(f"\nTransformer: {transformer_name}")
    print("Type:", type(transformer).__name__)
    print("Columns:", list(columns) if not isinstance(columns, str) else columns)

    if transformer == "drop" or transformer == "passthrough":
        continue

    if hasattr(transformer, "get_params"):
        print("Parameters:", transformer.get_params(deep=False))

try:
    transformed_names = preprocessor.get_feature_names_out()
    print("\nNumber of transformed features:", len(transformed_names))
    print("First 30 transformed feature names:")
    print(list(transformed_names[:30]))
except Exception as exc:
    print("\nTransformed feature names unavailable:", repr(exc))

print("\nNative regressor feature importance available:",
      hasattr(regressor, "feature_importances_"))
print("Native regressor coefficients available:",
      hasattr(regressor, "coef_"))


Preprocessor type: ColumnTransformer
Regressor type: HistGradientBoostingRegressor

Column transformer configuration:

Transformer: categorical
Type: OneHotEncoder
Columns: ['Crop', 'Season', 'Crop_Category', 'State', 'District']
Parameters: {'categories': 'auto', 'drop': None, 'dtype': <class 'numpy.float64'>, 'feature_name_combiner': 'concat', 'handle_unknown': 'ignore', 'max_categories': None, 'min_frequency': None, 'sparse_output': False}

Transformer: numeric
Type: FunctionTransformer
Columns: ['FE_YearStart']
Parameters: {'accept_sparse': True, 'check_inverse': False, 'feature_names_out': 'one-to-one', 'func': None, 'inv_kw_args': None, 'inverse_func': None, 'kw_args': None, 'validate': False}

Number of transformed features: 672
First 30 transformed feature names:
['categorical__Crop_Bajra', 'categorical__Crop_Barley', 'categorical__Crop_Gram', 'categorical__Crop_Jowar', 'categorical__Crop_Lentil', 'categorical__Crop_Maize', 'categorical__Crop_Moong', 'categorical__Crop_Ragi', '

In [6]:

# Restore required variables in case the notebook kernel was restarted.

import json
import joblib
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.inspection import permutation_importance

PROJECT_ROOT = Path(
    r"C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence"
)

DATA_PATH = (
    PROJECT_ROOT
    / "data/processed/weather_regional/apy_weather_integrated_exploratory.csv"
)
MODEL_DIR = PROJECT_ROOT / "models/yield_prediction"
MODEL_PATH = MODEL_DIR / "individual_crop_hgb_location.joblib"
MODEL_METADATA_PATH = MODEL_DIR / "individual_crop_hgb_location_metadata.json"
OUTPUT_DIR = PROJECT_ROOT / "models/model_explainability_and_scenarios"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

with open(MODEL_METADATA_PATH, "r", encoding="utf-8") as f:
    model_metadata = json.load(f)

df = pd.read_csv(DATA_PATH, low_memory=False)
model = joblib.load(MODEL_PATH)

feature_cols = model_metadata["features"]
target_col = model_metadata["target"]

# Filter individual-crop records and prepare model inputs.
explain_df = df[
    df["Crop_Category"].eq("Individual Crop Candidate")
].copy()

for col in ["Crop", "Season", "Crop_Category", "State", "District"]:
    explain_df[col] = explain_df[col].astype("string").str.strip()

explain_df["FE_YearStart"] = pd.to_numeric(
    explain_df["start_year"], errors="coerce"
)
explain_df[target_col] = pd.to_numeric(
    explain_df[target_col], errors="coerce"
)

explain_df = explain_df[
    explain_df[feature_cols].notna().all(axis=1)
    & explain_df[target_col].notna()
    & np.isfinite(explain_df["FE_YearStart"])
    & np.isfinite(explain_df[target_col])
    & (explain_df[target_col] >= 0)
].copy()

# Select the later-year diagnostic sample.
evaluation_df = explain_df[
    explain_df["FE_YearStart"].between(2021, 2022)
].copy()

MAX_EVAL_ROWS = 3000
if len(evaluation_df) > MAX_EVAL_ROWS:
    evaluation_df = evaluation_df.sample(
        n=MAX_EVAL_ROWS, random_state=42
    )

X_eval = evaluation_df[feature_cols].copy()
y_eval = evaluation_df[target_col].copy()

print("Dataset loaded:", df.shape)
print("Model loaded:", type(model).__name__)
print("Filtered observations:", len(explain_df))
print("Evaluation observations:", len(evaluation_df))
print("Evaluation years:",
      sorted(evaluation_df["FE_YearStart"].unique().tolist()))
print("\nTarget summary:")
print(y_eval.describe())

assert len(evaluation_df) > 0, "No evaluation observations found."
assert X_eval.notna().all().all(), "Missing model inputs remain."
assert np.isfinite(y_eval).all(), "Invalid target values remain."

print("\nEvaluation sample checks passed.")


Dataset loaded: (376272, 39)
Model loaded: Pipeline
Filtered observations: 159601
Evaluation observations: 3000
Evaluation years: [2021, 2022]

Target summary:
count     3000.000000
mean      1680.134333
std       1274.788181
min          0.000000
25%        768.750000
50%       1262.000000
75%       2360.500000
max      10550.000000
Name: Yield, dtype: float64

Evaluation sample checks passed.


In [7]:

from sklearn.inspection import permutation_importance
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Reuse the existing integrated dataset.
explain_df = df[
    df["Crop_Category"].eq("Individual Crop Candidate")
].copy()

feature_cols = model_metadata["features"]
target_col = model_metadata["target"]

# Normalize required fields and numeric values.
for col in ["Crop", "Season", "Crop_Category", "State", "District"]:
    explain_df[col] = explain_df[col].astype("string").str.strip()

explain_df["FE_YearStart"] = pd.to_numeric(
    explain_df["start_year"], errors="coerce"
)
explain_df[target_col] = pd.to_numeric(
    explain_df[target_col], errors="coerce"
)

explain_df = explain_df[
    explain_df[feature_cols].notna().all(axis=1)
    & explain_df[target_col].notna()
    & np.isfinite(explain_df["FE_YearStart"])
    & np.isfinite(explain_df[target_col])
    & (explain_df[target_col] >= 0)
].copy()

# Use the later available years for an out-of-time diagnostic.
evaluation_df = explain_df[
    explain_df["FE_YearStart"].between(2021, 2022)
].copy()

# Keep runtime manageable while preserving reproducibility.
MAX_EVAL_ROWS = 3000
if len(evaluation_df) > MAX_EVAL_ROWS:
    evaluation_df = evaluation_df.sample(
        n=MAX_EVAL_ROWS, random_state=42
    )

X_eval = evaluation_df[feature_cols].copy()
y_eval = evaluation_df[target_col].copy()

print("Filtered observations:", len(explain_df))
print("Evaluation observations:", len(evaluation_df))
print("Evaluation years:",
      sorted(evaluation_df["FE_YearStart"].unique().tolist()))
print("Feature columns:", X_eval.columns.tolist())
print("Target summary:")
print(y_eval.describe())

assert len(evaluation_df) > 0, "No evaluation observations found."
assert X_eval[feature_cols].notna().all().all()
assert np.isfinite(y_eval).all()

print("\nEvaluation sample checks passed.")


Filtered observations: 159601
Evaluation observations: 3000
Evaluation years: [2021, 2022]
Feature columns: ['Crop', 'Season', 'Crop_Category', 'State', 'District', 'FE_YearStart']
Target summary:
count     3000.000000
mean      1680.134333
std       1274.788181
min          0.000000
25%        768.750000
50%       1262.000000
75%       2360.500000
max      10550.000000
Name: Yield, dtype: float64

Evaluation sample checks passed.


In [8]:

# Generate predictions using the saved pipeline; no retraining.
y_pred = model.predict(X_eval)

evaluation_metrics = {
    "evaluation_rows": int(len(y_eval)),
    "MAE": float(mean_absolute_error(y_eval, y_pred)),
    "RMSE": float(np.sqrt(mean_squared_error(y_eval, y_pred))),
    "R2": float(r2_score(y_eval, y_pred)),
    "actual_mean": float(y_eval.mean()),
    "predicted_mean": float(np.mean(y_pred)),
    "actual_median": float(y_eval.median()),
    "predicted_median": float(np.median(y_pred)),
    "negative_predictions": int(np.sum(y_pred < 0)),
    "non_finite_predictions": int(np.sum(~np.isfinite(y_pred))),
}

print("Evaluation metrics:")
for metric, value in evaluation_metrics.items():
    print(f"{metric}: {value:.4f}" if isinstance(value, float)
          else f"{metric}: {value}")

comparison_df = evaluation_df[
    feature_cols + [target_col]
].copy()

comparison_df["Predicted_Yield"] = y_pred
comparison_df["Absolute_Error"] = np.abs(
    comparison_df[target_col] - comparison_df["Predicted_Yield"]
)
comparison_df["Prediction_Error"] = (
    comparison_df["Predicted_Yield"] - comparison_df[target_col]
)

print("\nSample predictions:")
display(comparison_df.head(10))

print("\nAbsolute error percentiles:")
print(comparison_df["Absolute_Error"].quantile(
    [0.50, 0.75, 0.90, 0.95, 0.99]
))


Evaluation metrics:
evaluation_rows: 3000
MAE: 424.0298
RMSE: 601.1638
R2: 0.7775
actual_mean: 1680.1343
predicted_mean: 1821.3452
actual_median: 1262.0000
predicted_median: 1519.4449
negative_predictions: 0
non_finite_predictions: 0

Sample predictions:


,Crop,Season,Crop_Category,State,District,FE_YearStart,Yield,Predicted_Yield,Absolute_Error,Prediction_Error
252346,Urad,Total,Individual Crop Candidate,Arunachal Pradesh,Leparada,2021,1034.0,624.216888,409.783112,-409.783112
300359,Rice,Kharif,Individual Crop Candidate,Mizoram,Mamit,2022,1355.0,1998.035408,643.035408,643.035408
305017,Bajra,Kharif,Individual Crop Candidate,Tamil Nadu,Chengalpattu,2022,2750.0,2208.535582,541.464418,-541.464418
312049,Maize,Kharif,Individual Crop Candidate,Uttar Pradesh,Unnao,2022,1994.0,2091.020089,97.020089,97.020089
287895,Urad,Total,Individual Crop Candidate,Chhattisgarh,Janjgir-Champa,2022,412.0,750.174246,338.174246,338.174246
261642,Jowar,Rabi,Individual Crop Candidate,Karnataka,Bagalkote,2021,1265.0,1217.215485,47.784515,-47.784515
294273,Urad,Total,Individual Crop Candidate,Karnataka,Mandya,2022,465.0,728.827945,263.827945,263.827945
267359,Maize,Kharif,Individual Crop Candidate,Maharashtra,Kolhapur,2021,3067.0,2703.021995,363.978005,-363.978005
276195,Ragi,Rabi,Individual Crop Candidate,Telangana,Nagarkurnool,2021,1875.0,1836.464426,38.535574,-38.535574
275262,Moong,Rabi,Individual Crop Candidate,Tamil Nadu,Vellore,2021,286.0,781.097942,495.097942,495.097942



Absolute error percentiles:
0.50     313.972587
0.75     573.695920
0.90     896.764934
0.95    1157.504022
0.99    2077.184572
Name: Absolute_Error, dtype: float64


In [9]:

# Permutation importance on the diagnostic evaluation sample.
# Scoring uses negative MAE, so larger importance means worse MAE
# after shuffling the feature.

perm_result = permutation_importance(
    model,
    X_eval,
    y_eval,
    scoring="neg_mean_absolute_error",
    n_repeats=3,
    random_state=42,
    n_jobs=-1
)

importance_df = pd.DataFrame({
    "feature": feature_cols,
    "importance_mean_mae_increase": perm_result.importances_mean,
    "importance_std": perm_result.importances_std
}).sort_values(
    "importance_mean_mae_increase",
    ascending=False
).reset_index(drop=True)

print("Permutation importance (increase in MAE):")
display(importance_df)

importance_output = OUTPUT_DIR / "permutation_feature_importance.csv"
importance_df.to_csv(importance_output, index=False)

print("\nSaved:", importance_output)
print(
    "\nInterpretation: a larger positive MAE increase indicates "
    "greater reliance on that feature for this evaluation sample."
)
print(
    "These are predictive associations, not causal effects. "
    "Correlated features can share or mask importance."
)


Permutation importance (increase in MAE):


,feature,importance_mean_mae_increase,importance_std
0,Crop,751.137951,16.014299
1,State,260.353835,3.511838
2,Season,47.556420,4.577723
3,District,36.506108,0.953568
4,FE_YearStart,4.598068,1.143711
5,Crop_Category,0.000000,0.000000



Saved: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\models\model_explainability_and_scenarios\permutation_feature_importance.csv

Interpretation: a larger positive MAE increase indicates greater reliance on that feature for this evaluation sample.
These are predictive associations, not causal effects. Correlated features can share or mask importance.


In [10]:

# Check categorical diversity in the evaluation sample.
categorical_cols = [
    "Crop", "Season", "Crop_Category", "State", "District"
]

print("Categorical feature diversity:")
for col in categorical_cols:
    print(f"\n{col}: {X_eval[col].nunique()} unique values")
    print(X_eval[col].value_counts(dropna=False).head(10))

print("\nSeason labels in the original individual-crop cohort:")
print(
    explain_df["Season"]
    .value_counts(dropna=False)
    .head(20)
)

# Check whether 'Total' is present in the evaluation data.
total_season_count = int(
    X_eval["Season"].astype("string").str.strip().str.casefold()
    .eq("total").sum()
)

print("\nEvaluation rows with Season='Total':", total_season_count)

# Save the categorical audit for reproducibility.
category_audit = {
    col: {
        "unique_values": int(X_eval[col].nunique()),
        "top_values": {
            str(key): int(value)
            for key, value in X_eval[col].value_counts().head(20).items()
        }
    }
    for col in categorical_cols
}

category_audit["evaluation_rows_with_total_season"] = total_season_count

audit_output = OUTPUT_DIR / "feature_category_audit.json"
with open(audit_output, "w", encoding="utf-8") as f:
    json.dump(category_audit, f, indent=2)

print("\nSaved audit:", audit_output)


Categorical feature diversity:

Crop: 13 unique values
Crop
Rice      406
Maize     374
Urad      328
Moong     305
Tur       252
Gram      228
Wheat     207
Bajra     205
Jowar     200
Lentil    137
Name: count, dtype: Int64

Season: 4 unique values
Season
Total     1240
Kharif     851
Rabi       675
Summer     234
Name: count, dtype: Int64

Crop_Category: 1 unique values
Crop_Category
Individual Crop Candidate    3000
Name: count, dtype: Int64

State: 34 unique values
State
Uttar Pradesh     414
Madhya Pradesh    245
Telangana         202
Karnataka         196
Tamil Nadu        182
Maharashtra       158
Bihar             152
Chhattisgarh      135
Assam             128
Gujarat           122
Name: count, dtype: Int64

District: 691 unique values
District
Gadag       14
Rewa        13
Koppal      13
Ariyalur    12
Mysuru      12
Meerut      12
Deoghar     11
Rajgarh     11
Raisen      11
Amethi      10
Name: count, dtype: Int64

Season labels in the original individual-crop cohort:
Seas

In [11]:

# Select a frequent, observed combination from the evaluation sample.
scenario_counts = (
    X_eval
    .groupby(
        ["Crop", "Season", "Crop_Category", "State", "District"],
        dropna=False
    )
    .size()
    .sort_values(ascending=False)
)

print("Most frequent observed combinations:")
display(scenario_counts.head(10).rename("row_count").reset_index())

# Choose the most frequent observed combination.
baseline_key = scenario_counts.index[0]

baseline_features = dict(zip(
    ["Crop", "Season", "Crop_Category", "State", "District"],
    baseline_key
))

# Use a year represented in the evaluation sample.
baseline_features["FE_YearStart"] = int(
    evaluation_df["FE_YearStart"].mode().iloc[0]
)

scenario_baseline = pd.DataFrame(
    [baseline_features],
    columns=feature_cols
)

baseline_prediction = float(model.predict(scenario_baseline)[0])

print("\nBaseline scenario inputs:")
display(scenario_baseline)

print(f"\nBaseline predicted yield: {baseline_prediction:.3f}")
print(
    "\nThis is a model prediction for an observed combination, "
    "not a measured yield or a forecast."
)


Most frequent observed combinations:


,Crop,Season,Crop_Category,State,District,row_count
0,Moong,Rabi,Individual Crop Candidate,West Bengal,North 24 Parganas,2
1,Moong,Kharif,Individual Crop Candidate,Nagaland,Kohima,2
2,Barley,Total,Individual Crop Candidate,Rajasthan,Sirohi,2
3,Barley,Total,Individual Crop Candidate,Uttar Pradesh,Amethi,2
4,Maize,Total,Individual Crop Candidate,Arunachal Pradesh,Lower Siang,2
5,Bajra,Kharif,Individual Crop Candidate,Andhra Pradesh,Visakhapatnam,2
6,Moong,Total,Individual Crop Candidate,Maharashtra,Kolhapur,2
7,Moong,Kharif,Individual Crop Candidate,Karnataka,Vijayapura,2
8,Maize,Rabi,Individual Crop Candidate,West Bengal,Birbhum,2
9,Jowar,Kharif,Individual Crop Candidate,Uttar Pradesh,Fatehpur,2



Baseline scenario inputs:


,Crop,Season,Crop_Category,State,District,FE_YearStart
0,Moong,Rabi,Individual Crop Candidate,West Bengal,North 24 Parganas,2022



Baseline predicted yield: 949.330

This is a model prediction for an observed combination, not a measured yield or a forecast.


In [12]:

# Compare years while holding all other features constant.
scenario_years = [2019, 2020, 2021, 2022, 2023, 2024]

scenario_rows = []
for year in scenario_years:
    row = baseline_features.copy()
    row["FE_YearStart"] = year
    scenario_rows.append(row)

scenario_df = pd.DataFrame(scenario_rows, columns=feature_cols)
scenario_df["Predicted_Yield"] = model.predict(
    scenario_df[feature_cols]
)

baseline_year = int(baseline_features["FE_YearStart"])
baseline_year_prediction = float(
    scenario_df.loc[
        scenario_df["FE_YearStart"].eq(baseline_year),
        "Predicted_Yield"
    ].iloc[0]
)

scenario_df["Change_vs_Baseline"] = (
    scenario_df["Predicted_Yield"] - baseline_year_prediction
)

scenario_df["Change_vs_Baseline_pct"] = np.where(
    baseline_year_prediction != 0,
    scenario_df["Change_vs_Baseline"] / baseline_year_prediction * 100,
    np.nan
)

print("Fixed scenario features:")
for col in ["Crop", "Season", "State", "District"]:
    print(f"{col}: {baseline_features[col]}")

print("\nHypothetical year comparison:")
display(
    scenario_df[
        [
            "FE_YearStart",
            "Predicted_Yield",
            "Change_vs_Baseline",
            "Change_vs_Baseline_pct"
        ]
    ].round(3)
)

scenario_output = OUTPUT_DIR / "year_scenario_comparison.csv"
scenario_df.to_csv(scenario_output, index=False)

print("\nSaved:", scenario_output)
print(
    "\nCaution: years outside the training period (2013–2022) are "
    "extrapolations in the year feature. Predictions for 2023–2024 "
    "must be treated especially cautiously."
)


Fixed scenario features:
Crop: Moong
Season: Rabi
State: West Bengal
District: North 24 Parganas

Hypothetical year comparison:


,FE_YearStart,Predicted_Yield,Change_vs_Baseline,Change_vs_Baseline_pct
0,2019,683.747,-265.583,-27.976
1,2020,734.619,-214.711,-22.617
2,2021,734.619,-214.711,-22.617
3,2022,949.330,0.000,0.000
4,2023,949.330,0.000,0.000
5,2024,949.330,0.000,0.000



Saved: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\models\model_explainability_and_scenarios\year_scenario_comparison.csv

Caution: years outside the training period (2013–2022) are extrapolations in the year feature. Predictions for 2023–2024 must be treated especially cautiously.


In [13]:

# Summarize errors by crop and season using the existing evaluation predictions.
error_analysis = comparison_df.copy()

error_analysis["Absolute_Error"] = np.abs(
    error_analysis[target_col] - error_analysis["Predicted_Yield"]
)

error_analysis["Squared_Error"] = (
    error_analysis[target_col] - error_analysis["Predicted_Yield"]
) ** 2

group_error_summary = (
    error_analysis
    .groupby(["Crop", "Season"], dropna=False)
    .agg(
        observations=(target_col, "size"),
        actual_median=(target_col, "median"),
        predicted_median=("Predicted_Yield", "median"),
        MAE=("Absolute_Error", "mean"),
        RMSE=("Squared_Error", lambda s: np.sqrt(s.mean()))
    )
    .reset_index()
)

# Avoid drawing strong conclusions from very small groups.
MIN_GROUP_SIZE = 20
group_error_summary["evidence_status"] = np.where(
    group_error_summary["observations"] >= MIN_GROUP_SIZE,
    "Minimum sample threshold met",
    "Small sample — interpret cautiously"
)

group_error_summary = group_error_summary.sort_values(
    ["observations", "MAE"],
    ascending=[False, False]
)

print("Largest crop-season groups by observation count:")
display(group_error_summary.head(25).round(3))

print("\nHighest-MAE crop-season groups with at least 20 observations:")
display(
    group_error_summary[
        group_error_summary["observations"] >= MIN_GROUP_SIZE
    ]
    .sort_values("MAE", ascending=False)
    .head(15)
    .round(3)
)

group_error_output = OUTPUT_DIR / "crop_season_error_analysis.csv"
group_error_summary.to_csv(group_error_output, index=False)

print("\nSaved:", group_error_output)


Largest crop-season groups by observation count:


,Crop,Season,observations,actual_median,predicted_median,MAE,RMSE,evidence_status
28,Rice,Kharif,142,2436.0,2376.470,410.734,581.661,Minimum sample threshold met
31,Rice,Total,138,2581.5,2690.034,449.217,600.743,Minimum sample threshold met
16,Maize,Kharif,135,2438.0,2412.386,515.633,670.980,Minimum sample threshold met
42,Urad,Total,125,659.0,729.195,259.745,342.290,Minimum sample threshold met
19,Maize,Total,123,2667.0,2909.347,715.261,939.115,Minimum sample threshold met
38,Tur,Total,123,978.0,1439.959,463.823,551.668,Minimum sample threshold met
9,Gram,Total,114,1234.0,1335.527,301.323,395.896,Minimum sample threshold met
23,Moong,Total,114,630.5,802.540,262.794,325.548,Minimum sample threshold met
8,Gram,Rabi,112,1197.5,1343.265,303.696,363.873,Minimum sample threshold met
36,Tur,Kharif,111,925.0,1052.955,308.317,380.577,Minimum sample threshold met



Highest-MAE crop-season groups with at least 20 observations:


,Crop,Season,observations,actual_median,predicted_median,MAE,RMSE,evidence_status
17,Maize,Rabi,69,3926.0,5116.172,926.823,1306.406,Minimum sample threshold met
3,Bajra,Total,90,1669.5,2128.005,734.123,885.573,Minimum sample threshold met
19,Maize,Total,123,2667.0,2909.347,715.261,939.115,Minimum sample threshold met
18,Maize,Summer,47,2878.0,2714.670,633.564,963.981,Minimum sample threshold met
13,Jowar,Total,75,1278.0,1533.714,583.504,902.007,Minimum sample threshold met
11,Jowar,Rabi,37,1245.0,1559.925,566.897,977.257,Minimum sample threshold met
25,Ragi,Rabi,20,1406.0,1868.917,561.655,661.700,Minimum sample threshold met
27,Ragi,Total,50,1342.5,1597.969,546.728,690.479,Minimum sample threshold met
29,Rice,Rabi,60,2882.0,2579.321,525.623,680.342,Minimum sample threshold met
16,Maize,Kharif,135,2438.0,2412.386,515.633,670.980,Minimum sample threshold met



Saved: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\models\model_explainability_and_scenarios\crop_season_error_analysis.csv


In [14]:

# Positive signed error means overprediction; negative means underprediction.
error_analysis["Signed_Error"] = (
    error_analysis["Predicted_Yield"] - error_analysis[target_col]
)

bias_summary = (
    error_analysis
    .groupby(["Crop", "Season"], dropna=False)
    .agg(
        observations=(target_col, "size"),
        mean_actual=(target_col, "mean"),
        mean_predicted=("Predicted_Yield", "mean"),
        mean_signed_error=("Signed_Error", "mean"),
        median_signed_error=("Signed_Error", "median"),
        MAE=("Absolute_Error", "mean")
    )
    .reset_index()
)

MIN_GROUP_SIZE = 20
bias_summary = bias_summary[
    bias_summary["observations"] >= MIN_GROUP_SIZE
].copy()

bias_summary["bias_direction"] = np.select(
    [
        bias_summary["mean_signed_error"] > 0,
        bias_summary["mean_signed_error"] < 0
    ],
    [
        "Overprediction on average",
        "Underprediction on average"
    ],
    default="No mean bias"
)

bias_summary["mean_bias_pct_of_actual"] = np.where(
    bias_summary["mean_actual"] != 0,
    bias_summary["mean_signed_error"] / bias_summary["mean_actual"] * 100,
    np.nan
)

print("Groups with the strongest average overprediction:")
display(
    bias_summary.sort_values("mean_signed_error", ascending=False)
    .head(10)
    .round(3)
)

print("\nGroups with the strongest average underprediction:")
display(
    bias_summary.sort_values("mean_signed_error", ascending=True)
    .head(10)
    .round(3)
)

bias_output = OUTPUT_DIR / "crop_season_prediction_bias.csv"
bias_summary.to_csv(bias_output, index=False)

print("\nSaved:", bias_output)
print(
    "\nInterpretation: signed error describes this evaluation sample; "
    "it does not establish the same bias in future data."
)


Groups with the strongest average overprediction:


,Crop,Season,observations,mean_actual,mean_predicted,mean_signed_error,median_signed_error,MAE,bias_direction,mean_bias_pct_of_actual
38,Tur,Total,123,957.496,1384.416,426.920,447.149,463.823,Overprediction on average,44.587
3,Bajra,Total,90,1747.678,2109.776,362.098,406.743,734.123,Overprediction on average,20.719
19,Maize,Total,123,3044.130,3358.880,314.750,373.533,715.261,Overprediction on average,10.340
27,Ragi,Total,50,1563.140,1861.980,298.840,262.491,546.728,Overprediction on average,19.118
6,Barley,Total,58,2415.500,2652.675,237.175,93.439,479.204,Overprediction on average,9.819
35,Small Millets,Total,58,927.586,1159.863,232.277,302.683,369.981,Overprediction on average,25.041
25,Ragi,Rabi,20,1850.300,2073.421,223.121,222.708,561.655,Overprediction on average,12.059
45,Wheat,Total,99,2532.788,2750.673,217.885,266.906,463.403,Overprediction on average,8.603
5,Barley,Rabi,51,2430.078,2646.949,216.870,100.383,414.502,Overprediction on average,8.924
31,Rice,Total,138,2599.174,2805.636,206.463,154.034,449.217,Overprediction on average,7.943



Groups with the strongest average underprediction:


,Crop,Season,observations,mean_actual,mean_predicted,mean_signed_error,median_signed_error,MAE,bias_direction,mean_bias_pct_of_actual
30,Rice,Summer,66,3216.682,3125.050,-91.632,-89.549,412.346,Underprediction on average,-2.849
11,Jowar,Rabi,37,1638.054,1560.065,-77.989,24.554,566.897,Underprediction on average,-4.761
29,Rice,Rabi,60,2811.183,2795.637,-15.547,-47.042,525.623,Underprediction on average,-0.553
41,Urad,Summer,34,839.265,850.058,10.793,12.158,165.916,Overprediction on average,1.286
16,Maize,Kharif,135,2682.674,2695.237,12.563,97.020,515.633,Overprediction on average,0.468
40,Urad,Rabi,62,640.984,663.471,22.487,95.711,224.128,Overprediction on average,3.508
10,Jowar,Kharif,84,1297.167,1329.414,32.248,27.720,410.163,Overprediction on average,2.486
18,Maize,Summer,47,3086.574,3130.514,43.939,-113.734,633.564,Overprediction on average,1.424
39,Urad,Kharif,107,662.617,707.137,44.520,48.397,222.539,Overprediction on average,6.719
28,Rice,Kharif,142,2446.127,2501.088,54.961,80.669,410.734,Overprediction on average,2.247



Saved: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\models\model_explainability_and_scenarios\crop_season_prediction_bias.csv

Interpretation: signed error describes this evaluation sample; it does not establish the same bias in future data.


In [15]:

# Separate positive and negative mean errors explicitly.
overprediction_groups = (
    bias_summary[bias_summary["mean_signed_error"] > 0]
    .sort_values("mean_signed_error", ascending=False)
    .reset_index(drop=True)
)

underprediction_groups = (
    bias_summary[bias_summary["mean_signed_error"] < 0]
    .sort_values("mean_signed_error", ascending=True)
    .reset_index(drop=True)
)

print("Top 10 groups by average overprediction:")
display(overprediction_groups.head(10).round(3))

print("\nTop 10 groups by average underprediction:")
display(underprediction_groups.head(10).round(3))

# Save separately so the rankings cannot be confused.
overprediction_groups.to_csv(
    OUTPUT_DIR / "top_overprediction_groups.csv", index=False
)
underprediction_groups.to_csv(
    OUTPUT_DIR / "top_underprediction_groups.csv", index=False
)

print("\nSaved corrected rankings:")
print(OUTPUT_DIR / "top_overprediction_groups.csv")
print(OUTPUT_DIR / "top_underprediction_groups.csv")


Top 10 groups by average overprediction:


,Crop,Season,observations,mean_actual,mean_predicted,mean_signed_error,median_signed_error,MAE,bias_direction,mean_bias_pct_of_actual
0,Tur,Total,123,957.496,1384.416,426.920,447.149,463.823,Overprediction on average,44.587
1,Bajra,Total,90,1747.678,2109.776,362.098,406.743,734.123,Overprediction on average,20.719
2,Maize,Total,123,3044.130,3358.880,314.750,373.533,715.261,Overprediction on average,10.340
3,Ragi,Total,50,1563.140,1861.980,298.840,262.491,546.728,Overprediction on average,19.118
4,Barley,Total,58,2415.500,2652.675,237.175,93.439,479.204,Overprediction on average,9.819
5,Small Millets,Total,58,927.586,1159.863,232.277,302.683,369.981,Overprediction on average,25.041
6,Ragi,Rabi,20,1850.300,2073.421,223.121,222.708,561.655,Overprediction on average,12.059
7,Wheat,Total,99,2532.788,2750.673,217.885,266.906,463.403,Overprediction on average,8.603
8,Barley,Rabi,51,2430.078,2646.949,216.870,100.383,414.502,Overprediction on average,8.924
9,Rice,Total,138,2599.174,2805.636,206.463,154.034,449.217,Overprediction on average,7.943



Top 10 groups by average underprediction:


,Crop,Season,observations,mean_actual,mean_predicted,mean_signed_error,median_signed_error,MAE,bias_direction,mean_bias_pct_of_actual
0,Rice,Summer,66,3216.682,3125.050,-91.632,-89.549,412.346,Underprediction on average,-2.849
1,Jowar,Rabi,37,1638.054,1560.065,-77.989,24.554,566.897,Underprediction on average,-4.761
2,Rice,Rabi,60,2811.183,2795.637,-15.547,-47.042,525.623,Underprediction on average,-0.553



Saved corrected rankings:
C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\models\model_explainability_and_scenarios\top_overprediction_groups.csv
C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\models\model_explainability_and_scenarios\top_underprediction_groups.csv


In [16]:

import json
from datetime import datetime, timezone

explainability_summary = {
    "project": "BharatCrop Intelligence",
    "analysis": "Model Explainability and Scenario Analysis",
    "model_name": model_metadata.get("model_name"),
    "target": target_col,
    "features": feature_cols,
    "evaluation": evaluation_metrics,
    "evaluation_period": {
        "year_start": int(evaluation_df["FE_YearStart"].min()),
        "year_end": int(evaluation_df["FE_YearStart"].max()),
        "evaluation_rows": int(len(evaluation_df)),
        "sampling_limit": MAX_EVAL_ROWS,
        "random_state": 42
    },
    "permutation_importance": importance_df.to_dict(orient="records"),
    "highest_mae_groups": (
        group_error_summary
        .sort_values("MAE", ascending=False)
        .head(10)[
            ["Crop", "Season", "observations", "MAE", "RMSE"]
        ]
        .to_dict(orient="records")
    ),
    "strongest_overprediction_groups": (
        overprediction_groups
        .head(10)[
            ["Crop", "Season", "observations",
             "mean_signed_error", "mean_bias_pct_of_actual"]
        ]
        .to_dict(orient="records")
    ),
    "strongest_underprediction_groups": (
        underprediction_groups
        .head(10)[
            ["Crop", "Season", "observations",
             "mean_signed_error", "mean_bias_pct_of_actual"]
        ]
        .to_dict(orient="records")
    ),
    "scenario_analysis": {
        "baseline_features": baseline_features,
        "baseline_prediction": baseline_prediction,
        "year_scenario_file": "year_scenario_comparison.csv",
        "caution": (
            "Years beyond 2022 are outside the model's documented "
            "training period and must not be treated as validated forecasts."
        )
    },
    "limitations": [
        "Permutation importance measures predictive reliance, not causation.",
        "Evaluation used a sample of up to 3000 observations from 2021–2022.",
        "The final holdout was repeatedly inspected during prior diagnostics.",
        "Crop-season group metrics can be unstable for small groups.",
        "The Total season label has not been established as directly comparable to individual seasons.",
        "The scenario baseline combination appeared only twice in the evaluation sample.",
        "Flat predictions for years beyond learned split thresholds do not imply constant real-world yield.",
        "Model performance does not establish production readiness."
    ],
    "generated_at_utc": datetime.now(timezone.utc).isoformat()
}

summary_path = OUTPUT_DIR / "explainability_summary.json"

with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(explainability_summary, f, indent=2, default=str)

print("Summary saved:", summary_path)
print("Summary sections:", list(explainability_summary.keys()))
print("Output directory:", OUTPUT_DIR)
print("\nFiles currently in output directory:")
for path in sorted(OUTPUT_DIR.iterdir()):
    if path.is_file():
        print(f"- {path.name} ({path.stat().st_size:,} bytes)")


Summary saved: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\models\model_explainability_and_scenarios\explainability_summary.json
Summary sections: ['project', 'analysis', 'model_name', 'target', 'features', 'evaluation', 'evaluation_period', 'permutation_importance', 'highest_mae_groups', 'strongest_overprediction_groups', 'strongest_underprediction_groups', 'scenario_analysis', 'limitations', 'generated_at_utc']
Output directory: C:\Users\gsr58\Desktop\Projects\BharatCrop-Intelligence\models\model_explainability_and_scenarios

Files currently in output directory:
- crop_season_error_analysis.csv (5,103 bytes)
- crop_season_prediction_bias.csv (5,604 bytes)
- explainability_summary.json (7,188 bytes)
- feature_category_audit.json (1,806 bytes)
- permutation_feature_importance.csv (305 bytes)
- top_overprediction_groups.csv (5,142 bytes)
- top_underprediction_groups.csv (596 bytes)
- year_scenario_comparison.csv (798 bytes)


# Conclusion: Model Explainability and Scenario Analysis

## 1. Objective Achieved
This notebook examined the existing `HistGradientBoostingRegressor` yield prediction pipeline using permutation feature importance, evaluation-sample error analysis, prediction-bias analysis, and controlled hypothetical scenarios. No model retraining was performed.

## 2. Key Findings

- **Feature importance:** Crop had the largest permutation importance, followed by State. Season and District had smaller measured contributions, while Crop_Category had zero importance in this evaluation sample because it was constant.
- **Evaluation performance:** On a sample of 3,000 observations from 2021–2022, the model achieved an MAE of approximately 424.03, RMSE of 601.16, and R² of 0.7775. These are sample-specific results, not a new independent validation.
- **Group-level errors:** Prediction errors varied across crop–season groups. Maize–Rabi had an MAE of approximately 926.82 across 69 observations.
- **Prediction bias:** Tur–Total showed the strongest average overprediction among the analyzed groups, with a mean signed error of approximately 426.92. Rice–Summer showed the strongest average underprediction, at approximately −91.63, among groups meeting the minimum sample threshold.
- **Year scenarios:** For the selected Moong–Rabi scenario in West Bengal's North 24 Parganas district, predicted yield increased from 683.747 in 2019 to 949.330 in 2022. Predictions remained unchanged through 2024, demonstrating a limitation of interpreting out-of-training-range scenarios as forecasts.

## 3. Generated Artifacts
Results were saved under `models/model_explainability_and_scenarios/`:

- `permutation_feature_importance.csv`
- `feature_category_audit.json`
- `year_scenario_comparison.csv`
- `crop_season_error_analysis.csv`
- `crop_season_prediction_bias.csv`
- `top_overprediction_groups.csv`
- `top_underprediction_groups.csv`
- `explainability_summary.json`

## 4. Limitations
Permutation importance measures predictive reliance, not causation. The evaluation sample was limited to 3,000 observations from 2021–2022, and the final holdout had been repeatedly inspected during earlier diagnostics. Some crop–season groups had limited observations. The meaning of the `Total` season category has not been established as directly comparable with individual seasons. The scenario baseline combination appeared only twice in the evaluation sample, and years beyond 2022 fall outside the documented training period.

These findings do not establish causal relationships, reliable future forecasts, or production readiness.

## 5. Final Assessment
The notebook provides a reproducible, artifact-based examination of model behavior and highlights areas requiring further validation. Its results should be used for model diagnostics and hypothesis generation, not as standalone agricultural recommendations.

**Status:** Explainability and scenario-analysis artifacts generated; independent validation and operational readiness remain separate requirements.